# Supply Chain & E-Commerce — Phase 1: Data Cleaning

**Dataset:** DataCo Supply Chain Dataset  
**Goal:** Load the raw flat CSV, clean it, and split it into 5 relational tables ready for PostgreSQL  
**Output:** 5 clean CSV files saved to `data/clean/`

---
## Tables we will produce
| Table | Key | Description |
|-------|-----|-------------|
| `customers` | Customer Id | Who bought |
| `products` | Product Card Id | What was sold |
| `orders` | Order Id | Transaction header |
| `order_items` | Order Item Id | Transaction line items |
| `shipping` | Order Id | Delivery & logistics info |

## 0. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Output directory for clean tables
os.makedirs('../data/clean', exist_ok=True)

print('Libraries loaded.')

Libraries loaded.


## 1. Load Raw Data

In [2]:
# Load the raw flat file
# Encoding latin-1 handles special characters common in this dataset
df = pd.read_csv('../data/DataCoSupplyChainDataset.csv', encoding='latin-1')

print(f'Raw dataset shape: {df.shape}')
print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]}')

Raw dataset shape: (180519, 53)
Rows: 180,519
Columns: 53


In [3]:
# First look at the data
df.head(3)

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class


In [4]:
# Column names and data types
df.dtypes

Type                              object
Days for shipping (real)           int64
Days for shipment (scheduled)      int64
Benefit per order                float64
Sales per customer               float64
Delivery Status                   object
Late_delivery_risk                 int64
Category Id                        int64
Category Name                     object
Customer City                     object
Customer Country                  object
Customer Email                    object
Customer Fname                    object
Customer Id                        int64
Customer Lname                    object
Customer Password                 object
Customer Segment                  object
Customer State                    object
Customer Street                   object
Customer Zipcode                 float64
Department Id                      int64
Department Name                   object
Latitude                         float64
Longitude                        float64
Market          

In [5]:
# Missing value overview
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)

print('Columns with missing values:')
print(missing_df)

Columns with missing values:
                     Missing Count  Missing %
Product Description         180519     100.00
Order Zipcode               155679      86.24
Customer Zipcode                 3       0.00
Customer Lname                   8       0.00


In [6]:
# Duplicate check
duplicates = df.duplicated().sum()
print(f'Fully duplicate rows: {duplicates}')

Fully duplicate rows: 0


## 2. Global Cleaning — Full DataFrame

Clean the flat file before splitting into tables.

In [7]:
# ── 2.1 Strip whitespace from all column names ────────────────────────────
df.columns = df.columns.str.strip()

# ── 2.2 Drop columns with no analytical value ─────────────────────────────
cols_to_drop = [
    'Customer Email',        # PII — no analytical value
    'Customer Password',     # PII — no analytical value
    'Product Description',   # Unstructured text — not useful for SQL/BI
    'Product Image',         # URL string — not useful
    'Order Item Cardprod Id' # Duplicate of Product Card Id
]

# Only drop columns that actually exist (defensive)
cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df.drop(columns=cols_to_drop, inplace=True)

print(f'Dropped {len(cols_to_drop)} columns: {cols_to_drop}')
print(f'Remaining columns: {df.shape[1]}')

Dropped 5 columns: ['Customer Email', 'Customer Password', 'Product Description', 'Product Image', 'Order Item Cardprod Id']
Remaining columns: 48


In [8]:
# ── 2.3 Parse date columns ────────────────────────────────────────────────
date_cols = ['order date (DateOrders)', 'Shipping date (DateOrders)']

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], infer_datetime_format=True, errors='coerce')
        nulls = df[col].isnull().sum()
        print(f'{col}: parsed. Unparseable rows: {nulls}')

order date (DateOrders): parsed. Unparseable rows: 0


In [9]:
# ── 2.4 Standardise string columns ───────────────────────────────────────
# Strip leading/trailing whitespace from all object columns
str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].apply(lambda col: col.str.strip())

# Standardise key categorical values to title case
cat_cols = [
    'Customer Segment', 'Delivery Status', 'Shipping Mode',
    'Order Status', 'Market', 'Department Name', 'Category Name'
]
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].str.title().str.strip()

print('String columns cleaned.')

String columns cleaned.


In [10]:
# ── 2.5 Remove fully duplicate rows ──────────────────────────────────────
before = len(df)
df.drop_duplicates(inplace=True)
after = len(df)
print(f'Duplicate rows removed: {before - after}')
print(f'Rows remaining: {after:,}')

Duplicate rows removed: 0
Rows remaining: 180,519


In [11]:
# ── 2.6 Validate numeric ranges ──────────────────────────────────────────

# Delivery risk flag should only be 0 or 1
if 'Late_delivery_risk' in df.columns:
    invalid_risk = df[~df['Late_delivery_risk'].isin([0, 1])]
    print(f'Invalid Late_delivery_risk values: {len(invalid_risk)}')

# Sales should be positive
if 'Sales' in df.columns:
    negative_sales = df[df['Sales'] < 0]
    print(f'Rows with negative Sales: {len(negative_sales)}')

# Order item quantity should be positive
if 'Order Item Quantity' in df.columns:
    zero_qty = df[df['Order Item Quantity'] <= 0]
    print(f'Rows with zero or negative quantity: {len(zero_qty)}')

# Discount rate should be between 0 and 1
if 'Order Item Discount Rate' in df.columns:
    invalid_discount = df[
        (df['Order Item Discount Rate'] < 0) |
        (df['Order Item Discount Rate'] > 1)
    ]
    print(f'Invalid discount rate rows: {len(invalid_discount)}')

Invalid Late_delivery_risk values: 0
Rows with negative Sales: 0
Rows with zero or negative quantity: 0
Invalid discount rate rows: 0


In [12]:
# ── 2.7 Validate delivery days logic ─────────────────────────────────────
# Real shipping days should be >= 0
if 'Days for shipping (real)' in df.columns:
    invalid_days = df[df['Days for shipping (real)'] < 0]
    print(f'Rows with negative real shipping days: {len(invalid_days)}')

# Scheduled days should be >= 0
if 'Days for shipment (scheduled)' in df.columns:
    invalid_sched = df[df['Days for shipment (scheduled)'] < 0]
    print(f'Rows with negative scheduled days: {len(invalid_sched)}')

Rows with negative real shipping days: 0
Rows with negative scheduled days: 0


In [13]:
# ── 2.8 Handle missing values ────────────────────────────────────────────
# Customer Zipcode — fill missing with 'Unknown'
if 'Customer Zipcode' in df.columns:
    n = df['Customer Zipcode'].isnull().sum()
    df['Customer Zipcode'] = df['Customer Zipcode'].fillna('Unknown')
    print(f'Customer Zipcode: filled {n} missing values with Unknown')

# Order Zipcode — same
if 'Order Zipcode' in df.columns:
    n = df['Order Zipcode'].isnull().sum()
    df['Order Zipcode'] = df['Order Zipcode'].fillna('Unknown')
    print(f'Order Zipcode: filled {n} missing values with Unknown')

# Product Status — fill with 0 (available) if missing
if 'Product Status' in df.columns:
    n = df['Product Status'].isnull().sum()
    df['Product Status'] = df['Product Status'].fillna(0).astype(int)
    print(f'Product Status: filled {n} missing values with 0')

Customer Zipcode: filled 3 missing values with Unknown
Order Zipcode: filled 155679 missing values with Unknown
Product Status: filled 0 missing values with 0


In [14]:
# ── 2.9 Final global shape check ─────────────────────────────────────────
print(f'Final flat DataFrame shape: {df.shape}')
print(f'Remaining nulls: {df.isnull().sum().sum()}')

Final flat DataFrame shape: (180519, 48)
Remaining nulls: 8


## 3. Split into Relational Tables

### 3.1 — customers

In [15]:
customer_cols = [
    'Customer Id',
    'Customer Fname',
    'Customer Lname',
    'Customer Segment',
    'Customer City',
    'Customer Country',
    'Customer State',
    'Customer Street',
    'Customer Zipcode',
    'Latitude',
    'Longitude'
]

# Only keep columns that exist in df
customer_cols = [c for c in customer_cols if c in df.columns]

customers = (
    df[customer_cols]
    .drop_duplicates(subset='Customer Id')
    .reset_index(drop=True)
)

print(f'customers shape: {customers.shape}')
print(f'Unique customers: {customers["Customer Id"].nunique():,}')
customers.head(3)

customers shape: (20652, 11)
Unique customers: 20,652


,Customer Id,Customer Fname,Customer Lname,Customer Segment,Customer City,Customer Country,Customer State,Customer Street,Customer Zipcode,Latitude,Longitude
0,20755,Cally,Holloway,Consumer,Caguas,Puerto Rico,PR,5365 Noble Nectar Island,725.0,18.251453,-66.037056
1,19492,Irene,Luna,Consumer,Caguas,Puerto Rico,PR,2679 Rustic Loop,725.0,18.279451,-66.037064
2,19491,Gillian,Maldonado,Consumer,San Jose,EE. UU.,CA,8510 Round Bear Gate,95125.0,37.292233,-121.881279


### 3.2 — products

In [16]:
product_cols = [
    'Product Card Id',
    'Product Name',
    'Product Price',
    'Product Category Id',
    'Category Id',
    'Category Name',
    'Department Id',
    'Department Name',
    'Product Status'
]

product_cols = [c for c in product_cols if c in df.columns]

products = (
    df[product_cols]
    .drop_duplicates(subset='Product Card Id')
    .reset_index(drop=True)
)

# Validate: product price should be positive
invalid_price = products[products['Product Price'] <= 0]
print(f'Products with invalid price: {len(invalid_price)}')

print(f'products shape: {products.shape}')
print(f'Unique products: {products["Product Card Id"].nunique():,}')
products.head(3)

Products with invalid price: 0
products shape: (118, 9)
Unique products: 118


,Product Card Id,Product Name,Product Price,Product Category Id,Category Id,Category Name,Department Id,Department Name,Product Status
0,1360,Smart watch,327.750000,73,73,Sporting Goods,2,Fitness,0
1,365,Perfect Fitness Perfect Rip Deck,59.990002,17,17,Cleats,4,Apparel,0
2,627,Under Armour Girls' Toddler Spine Surge Runni,39.990002,29,29,Shop By Sport,5,Golf,0


### 3.3 — orders

In [17]:
order_cols = [
    'Order Id',
    'Order Customer Id',
    'order date (DateOrders)',
    'Order Status',
    'Market',
    'Order Region',
    'Order City',
    'Order Country',
    'Order State',
    'Type'
]

order_cols = [c for c in order_cols if c in df.columns]

orders = (
    df[order_cols]
    .drop_duplicates(subset='Order Id')
    .reset_index(drop=True)
)

# Rename date column for cleanliness
orders.rename(columns={'order date (DateOrders)': 'order_date'}, inplace=True)

# Validate: orders with no matching customer
orphan_orders = orders[~orders['Order Customer Id'].isin(customers['Customer Id'])]
print(f'Orders with no matching customer: {len(orphan_orders)}')

# Check for null order dates
null_dates = orders['order_date'].isnull().sum()
print(f'Orders with null date: {null_dates}')

# Date range sanity check
print(f'Order date range: {orders["order_date"].min()} → {orders["order_date"].max()}')

print(f'orders shape: {orders.shape}')
orders.head(3)

Orders with no matching customer: 0
Orders with null date: 0
Order date range: 2015-01-01 00:00:00 → 2018-01-31 23:38:00
orders shape: (65752, 10)


,Order Id,Order Customer Id,order_date,Order Status,Market,Order Region,Order City,Order Country,Order State,Type
0,77202,20755,2018-01-31 22:56:00,Complete,Pacific Asia,Southeast Asia,Bekasi,Indonesia,Java Occidental,DEBIT
1,75939,19492,2018-01-13 12:27:00,Pending,Pacific Asia,South Asia,Bikaner,India,Rajastán,TRANSFER
2,75938,19491,2018-01-13 12:06:00,Closed,Pacific Asia,South Asia,Bikaner,India,Rajastán,CASH


### 3.4 — order_items

In [18]:
order_item_cols = [
    'Order Item Id',
    'Order Id',
    'Product Card Id',
    'Order Item Quantity',
    'Sales',
    'Order Item Total',
    'Order Item Discount',
    'Order Item Discount Rate',
    'Order Item Product Price',
    'Order Item Profit Ratio',
    'Benefit per order',
    'Order Profit Per Order',
    'Sales per customer'
]

order_item_cols = [c for c in order_item_cols if c in df.columns]

order_items = df[order_item_cols].copy().reset_index(drop=True)

# Validate: no negative sales
neg_sales = order_items[order_items['Sales'] < 0]
print(f'Rows with negative Sales: {len(neg_sales)}')

# Validate: no zero quantity
zero_qty = order_items[order_items['Order Item Quantity'] <= 0]
print(f'Rows with zero or negative quantity: {len(zero_qty)}')

# Validate: discount rate between 0 and 1
if 'Order Item Discount Rate' in order_items.columns:
    bad_disc = order_items[
        (order_items['Order Item Discount Rate'] < 0) |
        (order_items['Order Item Discount Rate'] > 1)
    ]
    print(f'Invalid discount rate rows: {len(bad_disc)}')

print(f'order_items shape: {order_items.shape}')
order_items.head(3)

Rows with negative Sales: 0
Rows with zero or negative quantity: 0
Invalid discount rate rows: 0
order_items shape: (180519, 13)


,Order Item Id,Order Id,Product Card Id,Order Item Quantity,Sales,Order Item Total,Order Item Discount,Order Item Discount Rate,Order Item Product Price,Order Item Profit Ratio,Benefit per order,Order Profit Per Order,Sales per customer
0,180517,77202,1360,1,327.75,314.640015,13.110000,0.04,327.75,0.29,91.250000,91.250000,314.640015
1,179254,75939,1360,1,327.75,311.359985,16.389999,0.05,327.75,-0.80,-249.089996,-249.089996,311.359985
2,179253,75938,1360,1,327.75,309.720001,18.030001,0.06,327.75,-0.80,-247.779999,-247.779999,309.720001


### 3.5 — shipping

In [19]:
shipping_cols = [
    'Order Id',
    'Shipping Mode',
    'Shipping date (DateOrders)',
    'Days for shipping (real)',
    'Days for shipment (scheduled)',
    'Delivery Status',
    'Late_delivery_risk'
]

shipping_cols = [c for c in shipping_cols if c in df.columns]

shipping = (
    df[shipping_cols]
    .drop_duplicates(subset='Order Id')
    .reset_index(drop=True)
)

# Rename date column for cleanliness
shipping.rename(columns={'Shipping date (DateOrders)': 'shipping_date'}, inplace=True)

# Engineered column: delivery delay (real - scheduled)
if 'Days for shipping (real)' in shipping.columns and 'Days for shipment (scheduled)' in shipping.columns:
    shipping['delivery_delay_days'] = (
        shipping['Days for shipping (real)'] -
        shipping['Days for shipment (scheduled)']
    )
    print('Engineered column: delivery_delay_days (positive = late, negative = early)')

# Delivery status value counts
print('\nDelivery Status distribution:')
print(shipping['Delivery Status'].value_counts())

# Shipping mode distribution
print('\nShipping Mode distribution:')
print(shipping['Shipping Mode'].value_counts())

print(f'\nshipping shape: {shipping.shape}')
shipping.head(3)

Engineered column: delivery_delay_days (positive = late, negative = early)

Delivery Status distribution:
Delivery Status
Late Delivery        36048
Advance Shipping     15127
Shipping On Time     11722
Shipping Canceled     2855
Name: count, dtype: int64

Shipping Mode distribution:
Shipping Mode
Standard Class    39324
Second Class      12778
First Class       10079
Same Day           3571
Name: count, dtype: int64

shipping shape: (65752, 7)


,Order Id,Shipping Mode,Days for shipping (real),Days for shipment (scheduled),Delivery Status,Late_delivery_risk,delivery_delay_days
0,77202,Standard Class,3,4,Advance Shipping,0,-1
1,75939,Standard Class,5,4,Late Delivery,1,1
2,75938,Standard Class,4,4,Shipping On Time,0,0


## 4. Summary — Before & After

In [20]:
summary = pd.DataFrame({
    'Table': ['customers', 'products', 'orders', 'order_items', 'shipping'],
    'Rows': [
        len(customers),
        len(products),
        len(orders),
        len(order_items),
        len(shipping)
    ],
    'Columns': [
        customers.shape[1],
        products.shape[1],
        orders.shape[1],
        order_items.shape[1],
        shipping.shape[1]
    ],
    'Nulls': [
        customers.isnull().sum().sum(),
        products.isnull().sum().sum(),
        orders.isnull().sum().sum(),
        order_items.isnull().sum().sum(),
        shipping.isnull().sum().sum()
    ]
})

print('=== CLEANING SUMMARY ===')
print(f'Raw flat file: {df.shape[0]:,} rows × {df.shape[1]} columns')
print()
print(summary.to_string(index=False))

=== CLEANING SUMMARY ===
Raw flat file: 180,519 rows × 48 columns

      Table   Rows  Columns  Nulls
  customers  20652       11      8
   products    118        9      0
     orders  65752       10      0
order_items 180519       13      0
   shipping  65752        7      0


## 5. Export Clean Tables

In [21]:
tables = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
    'shipping': shipping
}

for name, table in tables.items():
    path = f'../data/clean/{name}_clean.csv'
    table.to_csv(path, index=False)
    print(f'Saved: {path} — {len(table):,} rows')

print('\nAll clean tables exported successfully.')

Saved: ../data/clean/customers_clean.csv — 20,652 rows
Saved: ../data/clean/products_clean.csv — 118 rows
Saved: ../data/clean/orders_clean.csv — 65,752 rows
Saved: ../data/clean/order_items_clean.csv — 180,519 rows
Saved: ../data/clean/shipping_clean.csv — 65,752 rows

All clean tables exported successfully.


## 6. Next Step — Load into PostgreSQL

Use `02_load_to_postgres.ipynb` to:
1. Create the database schema in PostgreSQL
2. Load each clean CSV into its corresponding table
3. Set primary keys and foreign keys
4. Verify row counts match

